#### Imports

In [1]:
from pyspark.sql.functions import rand
from pyspark.sql.functions import input_file_name, regexp_extract
import os

StatementMeta(, a981249e-4036-4045-97ff-6306809e255d, 3, Finished, Available, Finished, False)

#### Parameters

In [2]:
# Split
TRAIN_RATIO = 0.8

# Tagged Parameters cell
image_size = ""   # default — overridden by pipeline @item()

StatementMeta(, a981249e-4036-4045-97ff-6306809e255d, 5, Finished, Available, Finished, False)

#### Read the entire dataframe regardless of folders

In [11]:
df = spark.read.format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load("Files/silver/resized/")

df.show()
# df.printSchema()
print("Shape: ", (df.count(), len(df.columns)))

StatementMeta(, a981249e-4036-4045-97ff-6306809e255d, 21, Finished, Available, Finished, False)

+--------------------+--------------------+------+--------------------+
|                path|    modificationTime|length|             content|
+--------------------+--------------------+------+--------------------+
|abfss://a96eab2a-...|2026-04-15 21:37:...| 60700|[FF D8 FF E0 00 1...|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 60011|[FF D8 FF E0 00 1...|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 59344|[FF D8 FF E0 00 1...|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 59243|[FF D8 FF E0 00 1...|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 59209|[FF D8 FF E0 00 1...|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 57067|[FF D8 FF E0 00 1...|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 55669|[FF D8 FF E0 00 1...|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 53993|[FF D8 FF E0 00 1...|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 46120|[FF D8 FF E0 00 1...|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 45389|[FF D8 FF E0 00 1...|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 43957|[FF D8 FF E0 0

In [12]:
# Filter to only the current size being processed by ForEach
df = df.filter(
    regexp_extract("path", r"resized/[^/]+/(\d+)/", 1) == str(image_size)
)

print(f"Filtered to size {image_size}: {df.count()} images")

StatementMeta(, a981249e-4036-4045-97ff-6306809e255d, 23, Finished, Available, Finished, False)

Filtered to size : 0 images


#### Get label & size

In [9]:
df = df.withColumn(
    "label",
    regexp_extract("path", r"resized/([^/]+)/\d+/", 1)  # ← updated regex
).withColumn(
    "size",
    regexp_extract("path", r"resized/[^/]+/(\d+)/", 1)
)

df.groupBy("label", "size").count().show()

StatementMeta(, a981249e-4036-4045-97ff-6306809e255d, 18, Finished, Available, Finished, False)

+------------+----+-----+
|       label|size|count|
+------------+----+-----+
|  gato_jorge| 512|   12|
|   gato_phil| 512|    3|
|perro_serena| 512|    5|
|  gato_jorge| 384|   12|
|   gato_phil| 384|    3|
|  gato_jorge| 256|   12|
|perro_serena| 384|    5|
|perro_serena| 256|    5|
|   gato_phil| 256|    3|
|  gato_jorge| 224|   12|
|   gato_phil| 224|    3|
|perro_serena| 224|    5|
|   gato_phil| 128|    5|
|  gato_jorge| 128|   12|
|perro_serena| 128|    5|
+------------+----+-----+



In [10]:
df.show()

StatementMeta(, a981249e-4036-4045-97ff-6306809e255d, 19, Finished, Available, Finished, False)

+--------------------+--------------------+------+--------------------+------------+----+
|                path|    modificationTime|length|             content|       label|size|
+--------------------+--------------------+------+--------------------+------------+----+
|abfss://a96eab2a-...|2026-04-15 21:37:...| 60700|[FF D8 FF E0 00 1...|  gato_jorge| 512|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 60011|[FF D8 FF E0 00 1...|  gato_jorge| 512|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 59344|[FF D8 FF E0 00 1...|  gato_jorge| 512|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 59243|[FF D8 FF E0 00 1...|  gato_jorge| 512|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 59209|[FF D8 FF E0 00 1...|  gato_jorge| 512|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 57067|[FF D8 FF E0 00 1...|  gato_jorge| 512|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 55669|[FF D8 FF E0 00 1...|  gato_jorge| 512|
|abfss://a96eab2a-...|2026-04-15 21:37:...| 53993|[FF D8 FF E0 00 1...|   gato_phil| 512|
|abfss://a

### Split generation (train & test)

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import rand, row_number, count, col

# Extract the original filename (the true identity of each image)

df = df.withColumn(
    "filename",
    regexp_extract("path", r"([^/]+)$", 1) 
)


# Get unique filenames per label and assign split there

filename_split = (
    df.select("filename","label")
    .distinct()
    .withColumn("row_num", row_number().over(
        Window.partitionBy("label").orderBy(rand())
    ))
    .withColumn("label_count",count("filename").over(
        Window.partitionBy("label")
    ))
    .withColumn(
        "split",
        (col("row_num")/col("label_count")>TRAIN_RATIO)
    )
    .select("filename","label","split")
)


# Join split decision back to full dataframe
df = df.join(filename_split.select("filename","split"), on="filename", how="left")

train_df = df.filter(col("split") == False)
test_df  = df.filter(col("split") == True)

In [ ]:
print("=== Train distribution ===")
train_df.groupBy("label").count().show()

print("=== Test distribution ===")
test_df.groupBy("label").count().show()

# Safety check — fail early if any label is missing from test
train_labels = {r["label"] for r in train_df.select("label").distinct().collect()}
test_labels  = {r["label"] for r in test_df.select("label").distinct().collect()}
missing      = train_labels - test_labels

if missing:
    raise ValueError(f"❌ These labels are missing from test set: {missing}. "
                     f"Increase dataset size or adjust TRAIN_RATIO.")
else:
    print("✅ All labels present in both train and test sets")

#### Image saving function

In [18]:
def save_images(df, base_path):
    rows = df.select("path", "content", "label").collect()

    for row in rows:
        label = row["label"]
        original_path = row["path"]
        content = row["content"]

        filename = os.path.basename(original_path)

        output_dir = f"/lakehouse/default/{base_path}/{label}"
        os.makedirs(output_dir, exist_ok=True)

        output_path = f"{output_dir}/{filename}"

        with open(output_path, "wb") as f:
            f.write(content)

StatementMeta(, 933a374d-93b4-4a4a-886b-323e576098c4, 27, Finished, Available, Finished, False)

#### Final saving into Gold layer

In [19]:
if image_size:
    base_output = f"Files/gold/dataset_{image_size}"
else:
    base_output = "Files/gold/dataset_all"

train_path = f"{base_output}/train"
test_path = f"{base_output}/test"

save_images(train_df, train_path)
save_images(test_df, test_path)

StatementMeta(, 933a374d-93b4-4a4a-886b-323e576098c4, 28, Finished, Available, Finished, False)

#### Quick validation

In [20]:
print("Train count:", train_df.count())
print("Test count:", test_df.count())

train_df.groupBy("label").count().show()
test_df.groupBy("label").count().show()


StatementMeta(, 933a374d-93b4-4a4a-886b-323e576098c4, 29, Finished, Available, Finished, False)

Train count: 77
Test count: 25
+------------+-----+
|       label|count|
+------------+-----+
|  gato_jorge|   43|
|perro_serena|   22|
|   gato_phil|   12|
+------------+-----+

+------------+-----+
|       label|count|
+------------+-----+
|  gato_jorge|   17|
|   gato_phil|    5|
|perro_serena|    3|
+------------+-----+

